# 03 — Feature Engineering

**Farms:** A · B · C &nbsp;|&nbsp; **Input:** per-farm `_avg_only.parquet` &nbsp;|&nbsp; **Output:** per-farm and combined `_features.parquet`

**Strategy:** sliding window over each event → extract mean, std, min, max per sensor column per window.

---

| Step | Description |
|------|-------------|
| 1 | Configuration |
| 2 | Load processed data |
| 3 | Feature extraction function |
| 4 | Process all farms |
| 5 | Combine, validate & save |


In [1]:
from pathlib import Path
from IPython.display import display

import pandas as pd
import numpy as np


## 1. Configuration


In [2]:
PROJECT_ROOT  = Path("..")
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

FARM_NAMES  = ["A", "B", "C"]
META_COLS   = {"time_stamp", "event_id", "label", "farm"}

WINDOW_SIZE = 24   # time steps per window
STEP_SIZE   = 24   # step between windows (non-overlapping)

print(f"Window size : {WINDOW_SIZE} steps")
print(f"Step size   : {STEP_SIZE} steps  ({'non-overlapping' if STEP_SIZE == WINDOW_SIZE else 'overlapping'})")


Window size : 24 steps
Step size   : 24 steps  (non-overlapping)


## 2. Load Processed Data


In [3]:
farm_dfs = {}

for farm_name in FARM_NAMES:
    path = PROCESSED_DIR / f"windfarm{farm_name}_avg_only.parquet"
    df = pd.read_parquet(path)
    farm_dfs[farm_name] = df
    print(f"Farm {farm_name}: {df.shape[0]:,} rows × {df.shape[1]} cols")


Farm A: 1,196,747 rows × 50 cols
Farm B: 859,065 rows × 67 cols
Farm C: 3,187,136 rows × 242 cols


In [4]:
farm_sensor_cols = {}

print(f"{'Farm':<6} {'Sensor cols':>12}  {'Events (normal / anomaly)'}")
print("-" * 52)

for farm_name, df in farm_dfs.items():
    s_cols = [c for c in df.columns if c not in META_COLS]
    farm_sensor_cols[farm_name] = s_cols
    dist = df.groupby("label")["event_id"].nunique().to_dict()
    normal  = dist.get("normal",  0)
    anomaly = dist.get("anomaly", 0)
    print(f"  {farm_name:<4} {len(s_cols):>12}  {normal} normal / {anomaly} anomaly")


Farm    Sensor cols  Events (normal / anomaly)
----------------------------------------------------
  A              46  11 normal / 11 anomaly
  B              63  9 normal / 6 anomaly
  C             238  31 normal / 27 anomaly


## 3. Feature Extraction Function


In [5]:
def extract_window_features(window_df, sensor_cols):
    """Compute mean, std, min, max for every sensor column in the window."""
    features = {}
    for col in sensor_cols:
        features[f"{col}_mean"] = window_df[col].mean()
        features[f"{col}_std"]  = window_df[col].std()
        features[f"{col}_min"]  = window_df[col].min()
        features[f"{col}_max"]  = window_df[col].max()
    return features


## 4. Process All Farms


In [6]:
all_samples = []

for farm_name, df in farm_dfs.items():
    sensor_cols = farm_sensor_cols[farm_name]
    farm_samples = []

    for event_id, event_df in df.groupby("event_id"):
        event_df = event_df.sort_values("time_stamp").reset_index(drop=True)
        label    = event_df["label"].iloc[0]

        for start in range(0, len(event_df) - WINDOW_SIZE, STEP_SIZE):
            window = event_df.iloc[start : start + WINDOW_SIZE]
            row = extract_window_features(window, sensor_cols)
            row["event_id"] = event_id
            row["label"]    = label
            row["farm"]     = farm_name
            farm_samples.append(row)

    print(f"Farm {farm_name}: {len(farm_samples):,} windows from {df['event_id'].nunique()} events")
    all_samples.extend(farm_samples)

print(f"\nTotal windows across all farms: {len(all_samples):,}")


Farm A: 49,851 windows from 22 events
Farm B: 35,784 windows from 15 events
Farm C: 132,779 windows from 58 events

Total windows across all farms: 218,414


In [7]:
features_df = pd.DataFrame(all_samples)

print(f"Features shape: {features_df.shape[0]:,} windows × {features_df.shape[1]} columns")
display(features_df.head())


Features shape: 218,414 windows × 1019 columns


,sensor_0_avg_mean,sensor_0_avg_std,sensor_0_avg_min,sensor_0_avg_max,sensor_1_avg_mean,sensor_1_avg_std,sensor_1_avg_min,sensor_1_avg_max,sensor_2_avg_mean,sensor_2_avg_std,...,wind_speed_236_avg_min,wind_speed_236_avg_max,wind_speed_235_avg_mean,wind_speed_235_avg_std,wind_speed_235_avg_min,wind_speed_235_avg_max,wind_speed_237_avg_mean,wind_speed_237_avg_std,wind_speed_237_avg_min,wind_speed_237_avg_max
0,21.916667,0.408248,21.0,23.0,186.308333,73.241013,66.2,340.6,12.829167,73.233999,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,26.750000,1.870829,23.0,29.0,266.445833,32.991501,188.4,324.7,17.458333,29.804375,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,29.875000,0.448427,29.0,31.0,266.795833,16.985044,210.7,292.4,-1.283333,18.349877,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,27.583333,1.017955,26.0,30.0,275.262500,8.612822,263.0,295.8,1.162500,10.390079,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,25.750000,0.442326,25.0,26.0,210.883333,45.533787,149.3,284.4,-17.579167,46.364491,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 5. Combine, Validate & Save


In [8]:
features_df["target"] = features_df["label"].map({"normal": 0, "anomaly": 1})

print("Windows per farm & label:")
display(
    features_df.groupby(["farm", "label"])
    .agg(windows=("event_id", "count"), unique_events=("event_id", "nunique"))
)


Windows per farm & label:


windows  unique_events
farm label                          
A    anomaly    24800             11
     normal     25051             11
B    anomaly    14719              6
     normal     21065              9
C    anomaly    61996             27
     normal     70783             31

In [9]:
print(f"Target distribution:")
print(features_df["target"].value_counts().rename({0: "normal", 1: "anomaly"}).to_string())


Target distribution:
target
normal     116899
anomaly    101515


In [10]:
total_nans = features_df.isna().sum().sum()
print(f"Total NaNs: {total_nans:,}")
print("(NaNs arise for sensor columns that exist in one farm but not another — filled with 0 below)")


Total NaNs: 77,312,864
(NaNs arise for sensor columns that exist in one farm but not another — filled with 0 below)


In [11]:
features_df = features_df.fillna(0)
print(f"NaNs after fill: {features_df.isna().sum().sum()}")


NaNs after fill: 0


In [12]:
# Save combined features
combined_path = PROCESSED_DIR / "all_farms_features.parquet"
features_df.to_parquet(combined_path, index=False)
print(f"Saved combined : {combined_path.name}  ({features_df.shape[0]:,} windows × {features_df.shape[1]} features)")

# Save per-farm features
for farm_name, group in features_df.groupby("farm"):
    path = PROCESSED_DIR / f"windfarm{farm_name}_features.parquet"
    group.to_parquet(path, index=False)
    print(f"Saved Farm {farm_name}   : {path.name}  ({group.shape[0]:,} windows)")


Saved combined : all_farms_features.parquet  (218,414 windows × 1020 features)
Saved Farm A   : windfarmA_features.parquet  (49,851 windows)
Saved Farm B   : windfarmB_features.parquet  (35,784 windows)
Saved Farm C   : windfarmC_features.parquet  (132,779 windows)
